# Memmingen -- Zentrale vs. dezentrale Pumpstation (Druckuntersuchung)

**Standalone config, unabhängig von Paper 2's Szenario-Matrix.** Lädt
`configs/pressure/Memmingen_pressure_stations.yaml` direkt -- keine Abhängigkeit
von `scripts/paper_2/scenarios.yaml` oder `scenario_runner.py`s Szenario-Merge
(Heizkurve, HP-Platzierung, DSM, TES-Override sind alle bereits in der YAML
selbst gesetzt). Physik-Referenz für alles Druck/Pumpen-bezogene:
`docs/paper_2/CALION_Paper2_Implementation_Statement.md` Part H.

**Neu (2026-07-29): reale Übergabestations-Anzahl pro Knoten.** Die DXF
(`configs/memmingen/2026-07-06_MM-Nord_Bestand_FW.dxf`) zeigt 174 echte
Gebäudeeinführungen (120 Wohngebäude + 54 Industriebauten) -- das Modell
bildet diese in nur 15 Knoten / 27 V_x-Zonen ab. `Memmingen_pressure_stations.yaml`
fügt pro Knoten `n_transfer_stations` hinzu (demand-proportionale Schätzung,
siehe YAML-Kopfkommentar für Methodik + bekannte Grenzen). Ändert NICHTS an
der gelösten Physik (aggregierte `station_{node}_P_pump` ist
mathematisch unabhängig von der Stationsanzahl bei gleichem Δp pro Station --
siehe Cell 10/11) -- macht aber sichtbar, WIE VIELE reale Stationen sich
hinter einem einzigen `min_required_bar`-Wert verstecken (bis zu 35 an j_12).

**Neu (2026-07-29, selbe Session): reale Zuleitungs-Druckverluste
("last mile" von Knoten zu Übergabestation).** Bisher komplett unmodelliert
-- nur der stationsinterne `delta_p_min_consumer_bar` (0.6 bar) wurde
verrechnet, nicht das Rohrstück DAHIN. Aus der DXF (Vorlauf + Doppelrohrleitung-
Layer kombiniert -> 80.5% ein zusammenhängendes Graph statt vorher 58%)
zurückverfolgt: reale Zuleitungslängen median 15m, p90 67m, DN25-32 dominant.
Pro Knoten wird die ERWARTETE LÄNGSTE Zuleitung bootstrap-geschätzt (mehr
`n_transfer_stations` -> statistisch wahrscheinlicher, dass eine der langen
DXF-Zuleitungen dabei ist -- ohne dass man wissen muss, WELCHE realen Gebäude
hinter welchem Knoten liegen). Modelliert als konvexe (Δp~Durchfluss²)
gewichtete Stützstellen-Kombination OHNE Binärvariablen -- exakt (nicht nur
eine Schranke), weil sie ausschließlich in einer `>=`-Druckanforderung und in
einem vom Solver minimierten MW-Term auftaucht (siehe
`calion/models/blocks/thermal_node.py`s neuer Block + memory
project_memmingen_pump_pressure_study für die volle Herleitung). Standardmäßig
AUS für jede Config ohne `pressure.lateral_length_m` -- komplett rückwärtskompatibel,
bytegleich verifiziert an `Memmingen_pressure.yaml`.

Vergleicht zwei Varianten des **unveränderten** Memmingen-Netzes (15 Knoten,
14 Rohre, echte DXF-Topologie, `primary_producer` = `j_9` = die reale
Energiezentrale, siehe `configs/pressure/Memmingen_componenets_spec`):

- **Szenario A ("central")**: nur die zentrale Pumpe am Erzeuger `j_9`
  (Setpoint aus der YAML, siehe dort für die reale Wilo-Pumpenkurven-Herleitung).
- **Szenario B ("central_plus_j13")**: zusätzlich eine zweite Pumpstation am
  Knoten `j_13` (10 MW_th Generator, eigener Druck-Floor) -- macht
  `j_13` automatisch zu einem `producer`/`mixed`-Knoten; die im Framework
  bereits vorhandene Sekundär-Pumpstations-Logik greift ohne Codeänderung
  (siehe Part H.8 "How to add a pump station yourself").

Ruft die **echte** `calion/`-Pipeline auf (kein Nachbau). Einzige verbleibende
Abhängigkeit von `scripts/paper_2/`: ein paar generische Utility-Funktionen
(`_load_yaml`, `_apply_spatial_temperature_offsets`, COP-Precompute-Helfer) --
das sind Framework-Helfer, keine Paper-2-Szenariodaten.

**Explizite Annahmen (jetzt in der YAML selbst, nicht mehr im Notebook-Code):**
- `network.primary_producer` bestimmt den Referenzknoten (aktuell `j_9`) --
  dieses Notebook liest ihn dynamisch aus der YAML, statt einen Knotennamen
  fest zu verdrahten (genau das brach zuletzt, als die Erzeugung von `j_1`
  nach `j_9` verschoben wurde).
- `network.nodes.<primary_producer>.pressure.setpoint_bar` -- Referenzdruck.
- `network.nodes.<jeder reine Verbraucherknoten>.pressure.min_required_bar = 2.0`
- `network.physics.pressure_regularization = true` (Part H.7 -- macht Knotendrücke
  downstream vom fixen Erzeuger physikalisch aussagekräftig statt
  solver-beliebig; verifiziert kostenneutral auf diesem Netz)
- Zweite Pumpstation an `j_13` (nur Szenario B, hier im Notebook gesetzt,
  NICHT in der YAML -- das ist genau der Unterschied zwischen A und B).

**Wichtig (2026-07-29):** `network.nodes.<id>.pressure.{setpoint_bar,
min_required_bar}` wurde bis eben campaign-weit von `model_finalizer.py`
stillschweigend verworfen (fehlte in dessen Feld-Whitelist) -- jeder
Nicht-Standard-Wert wurde durch den Code-Default (10.0 bar / keine Pflicht)
ersetzt, ohne Fehlermeldung. Jetzt gefixt und verifiziert; siehe
Implementation Statement Part H.2 für Details.


### Cell 1 — Setup: Pfade + Imports

In [ ]:
# =============================================================================
# Cell 1 — Setup: path bootstrap + imports
# =============================================================================
import copy
import math as _math
import sys, json, time
from pathlib import Path

_ROOT = Path(r"c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat")
sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo

# Generic framework utilities only -- NOT scenario data. No scenarios.yaml,
# no load_scenarios_config(), no scen/scen_cfg dicts anywhere in this notebook.
from scripts.paper_2.scenario_runner import (
    _load_yaml, _dump_yaml_tmp, _apply_spatial_temperature_offsets,
    _inject_cop_series, _load_outdoor_temps, _load_hp_source_temps,
)
from calion.run.workflow import _build_workflow_inputs
from calion.utils.heizkurve import compute_heizkurve
from calion.utils.cop_wrapper import precompute_cop
from calion.models.system_builder import build_model
from calion.io.thermal_network_exporter import export_all_results

print("Setup OK. Repo root:", _ROOT)


### Cell 2 — Config laden (standalone, keine Szenario-Datei)

In [ ]:
# =============================================================================
# Cell 2 — Load the standalone config directly
# =============================================================================
CONFIG_PATH = _ROOT / "configs" / "pressure" / "Memmingen_pressure_stations.yaml"
cfg = _load_yaml(CONFIG_PATH)

# Read dynamically, not hardcoded -- the primary producer moved from j_1 to
# j_9 once already (2026-07-29) when generation was relocated to match the
# real Energiezentrale. Everything below keys off this variable instead of a
# literal node name so the notebook survives the next topology edit too.
PRIMARY_PRODUCER = cfg["network"]["primary_producer"]

print("Config loaded from:", CONFIG_PATH)
print("primary_producer:", PRIMARY_PRODUCER)
print(f"{PRIMARY_PRODUCER} pressure setpoint:",
      cfg["network"]["nodes"][PRIMARY_PRODUCER]["pressure"]["setpoint_bar"])
print("pressure_regularization:", cfg["network"]["physics"].get("pressure_regularization"))
print("j_13 node (before any change):", cfg["network"]["nodes"]["j_13"])

n_stations_total = sum(n.get("n_transfer_stations", 0) for n in cfg["network"]["nodes"].values())
print(f"\nn_transfer_stations sum across all nodes: {n_stations_total} "
      "(should be 174 -- real DXF Gebaeudeeinfuehrungen count)")


### Cell 3 — Zeitfenster begrenzen (Januar = kältester/lastreichster Monat)

In [ ]:
# =============================================================================
# Cell 3 — Restrict horizon to January (worst-case peak-demand window).
# Full year is the YAML default (see scenario.horizon there) -- shortened here
# for a tractable interactive solve. Raise/lower as needed.
# =============================================================================
cfg["scenario"]["horizon"] = {"start": "2025-01-01 00:00", "end": "2025-01-31 23:00"}
print("Horizon restricted to:", cfg["scenario"]["horizon"])


### Cell 4 — Bereits in der YAML gesetzte Druckstudie-Annahmen (nur zur Kontrolle)

In [ ]:
# =============================================================================
# Cell 4 — Nothing is SET here anymore (all baked into Memmingen_pressure.yaml)
# -- just surfacing what's active for visibility before the solve.
# =============================================================================
PURE_CONSUMER_NODES = [
    nid for nid, ncfg in cfg["network"]["nodes"].items() if not ncfg.get("assets")
]
print("Pure consumer nodes (from YAML, no assets attached):", sorted(PURE_CONSUMER_NODES))
for nid in sorted(PURE_CONSUMER_NODES):
    print(f"  {nid}: min_required_bar =",
          cfg["network"]["nodes"][nid].get("pressure", {}).get("min_required_bar"),
          "| n_transfer_stations =", cfg["network"]["nodes"][nid].get("n_transfer_stations"))

print("\nAll nodes by n_transfer_stations (descending -- aggregation-risk ranking):")
for nid, ncfg in sorted(cfg["network"]["nodes"].items(),
                         key=lambda kv: -(kv[1].get("n_transfer_stations", 0))):
    print(f"  {nid}: n_transfer_stations={ncfg.get('n_transfer_stations', 0):3d}  "
          f"({len(ncfg.get('consumers', []))} V_x column(s))")


### Cell 5 — Gemeinsame Pipeline: Tabelle laden, Heizkurve, COP, räumliche T-Offsets

Heizkurven-Parameter (`k`, `T_supply_min_c`, `T_supply_max_c`) kommen jetzt direkt aus `cfg["network"]["heating_curve"]` -- keine `heat_curve_stages`-Lookup mehr. Diese Schritte hängen nur von Topologie/Nachfrage/Wetter ab, nicht davon ob j_13 einen Kessel hat -- deshalb einmal auf dem gemeinsamen `cfg` berechnet und danach für beide Szenarien wiederverwendet.

In [ ]:
# =============================================================================
# Cell 5 — Shared pipeline (table, heating curve, COP, spatial offsets),
# computed ONCE on the shared cfg before branching into Scenario A / B.
# =============================================================================
tmp1 = _dump_yaml_tmp(cfg)
inputs0 = _build_workflow_inputs([str(tmp1)], overrides=None)
table = inputs0.table
try:
    tmp1.unlink()
except OSError:
    pass
print("Table rows:", len(table), "| dt_h:", inputs0.dt_h)

hc = cfg["network"]["heating_curve"]
T_aus = _load_outdoor_temps(table, cfg)
T_VL_ts = compute_heizkurve(
    k=hc["k"], T_VL_min_c=hc["T_supply_min_c"],
    T_VL_max_c=hc["T_supply_max_c"], T_aus_ts=T_aus,
)
return_temp_c = float(cfg.get("network", {}).get("return_temp_c", 60.0))
min_delta_T = float(cfg.get("network", {}).get("min_supply_delta_T_k", 10.0))
T_VL_min_effective = max(float(hc["T_supply_min_c"]), return_temp_c + min_delta_T)
T_VL_ts = np.maximum(T_VL_ts, T_VL_min_effective)
cfg["network"]["heating_curve"]["T_supply_min_c"] = T_VL_min_effective
cfg.setdefault("heat_pumps", {}).setdefault("cop", {})
cfg["heat_pumps"]["cop"]["supply_temp_min_c"] = T_VL_min_effective
cfg["heat_pumps"]["cop"]["supply_temp_max_c"] = hc["T_supply_max_c"]

delta_T_scenario_k = round(T_VL_min_effective - return_temp_c, 2)
for _ak, _acfg in cfg.get("assets", {}).items():
    if _acfg.get("type") == "geometric_storage":
        _acfg["delta_T_scenario_k"] = delta_T_scenario_k

T_source_ts = _load_hp_source_temps(cfg, table)
cop_ts = precompute_cop(T_VL_ts=T_VL_ts, T_source_ts=T_source_ts, table=table, cfg=cfg, hp_type="standard")
_inject_cop_series(cfg, cop_ts)

_apply_spatial_temperature_offsets(cfg, T_VL_ts, "MM-PRESSURE-STUDY")
offsets = {nid: ncfg.get("T_supply_offset_c") for nid, ncfg in cfg["network"]["nodes"].items() if "T_supply_offset_c" in ncfg}
print("Node T_supply_offset_c:", offsets)
print("\nCELLS 1-5 (shared pipeline) OK")


### Cell 6 — Szenario A / B abzweigen: j_13-Gaskessel + zweite Pumpstation nur in B

In [ ]:
# =============================================================================
# Cell 6 — Branch into Scenario A (central only) and Scenario B (central + j_13
# pump station). Everything upstream (topology, demand, heating curve, COP,
# spatial offsets, consumer min-pressure floors, regularization) is IDENTICAL
# in both, already baked into the YAML -- the only difference is the j_13
# asset + its own pump setpoint, set here in Python.
# =============================================================================
PUMP_SETPOINT_J13_BAR = 10.0  # symmetric w/ the primary producer -- see notebook intro for rationale

cfg_A = copy.deepcopy(cfg)   # Scenario A: "central" -- j_13 stays a pure consumer

cfg_B = copy.deepcopy(cfg)   # Scenario B: "central_plus_j13"
cfg_B["assets"]["gasboiler_j13"] = {
    "type": "thermal_generator",
    "fuel": "gas",
    "capacity_mw": 3.0,
    "thermal_efficiency": 0.90,
    "min_load": 0.1,
}
cfg_B["network"]["nodes"]["j_13"]["assets"] = ["gasboiler_j13"]
cfg_B["network"]["nodes"]["j_13"].setdefault("pressure", {})["setpoint_bar"] = PUMP_SETPOINT_J13_BAR

print("Scenario A j_13:", cfg_A["network"]["nodes"]["j_13"])
print("Scenario B j_13:", cfg_B["network"]["nodes"]["j_13"])


### Cell 6b — Notebook-lokaler Fix: Tangentenpunkte bei niedrigem Durchfluss (nur noch `delta_p`)

`pipe_pair.py`s `delta_p` nutzt weiterhin eine 3-Punkt konvexe Tangenten-Unterhülle,
verankert bei 33%/67%/100% der Rohr-Auslegungskapazität (`effective_max_flow`, aus
`max_velocity_m_s`). Eine Tangente wird erst oberhalb von ~2/3 ihres Ankerpunkts
positiv/bindend -- bei echten Januar-Durchflüssen (oft nur 5-12% der Auslegung)
bleiben ALLE 3 Tangenten nicht-bindend, wodurch `delta_p` vom Solver
frei/unphysikalisch gesetzt werden kann (siehe Implementation Statement Part H.5).

**Nur für dieses Notebook** (nicht `pipe_pair.py`, nicht die laufende Paper-2-
Kampagne): zusätzliche Tangentenpunkte bei 2/5/8/12/18/25% der Auslegungskapazität,
nachträglich auf das bereits gebaute Modell gestapelt (gleiche Formel wie
`pipe_pair.py`, nur zusätzliche `>=`-Constraints -- verschärft nur, ändert nichts
an der ursprünglichen zulässigen Menge).

**2026-07-29: `P_pump` aus dieser Funktion entfernt.** `pipe_pair.py`s eigener
Standard (`pump_pin_pwl=true`, Implementation Statement H.6.5) pinnt `P_pump`
inzwischen selbst auf eine PWL-Gleichung der echten Kubik -- genau das Problem,
das diese Funktion für `P_pump` vorher notebook-lokal geflickt hat, ist damit
bereits campaign-weit gelöst. Zusätzliche `P_pump >= tangente(...)`-Schranken
oben drauf wären bestenfalls redundant; schlimmstenfalls könnte ein Tangentenpunkt,
dessen Sekantenwert die PWL-Gleichung an diesem Durchfluss übersteigt (hier nicht
verifiziert ausgeschlossen), das Modell infeasible machen. `delta_p` ist von H.6.5
nicht betroffen (bleibt tangenten-gebunden, siehe H.6.5: "Δp still uses tangents"),
daher hier weiterhin sinnvoll verdichtet.

In [ ]:
# =============================================================================
# Cell 6b — add_low_flow_tangents(): notebook-local tightening, NOT a pipe_pair.py
# or campaign-wide change. Densifies the SAME convex tangent envelope toward the
# low-flow end so delta_p tracks real physics at this network's actual (well
# below design-capacity) January flows.
#
# 2026-07-29: P_pump is DELIBERATELY no longer touched here (was removed). See
# the Cell 6b markdown above for why -- pipe_pair.py's own pump_pin_pwl=true
# default (Implementation Statement H.6.5) now pins P_pump to a PWL equality,
# making this function's old P_pump tangent lower-bounds redundant/risky.
# =============================================================================
def add_low_flow_tangents(model, network_manager, extra_fracs=(0.02, 0.05, 0.08, 0.12, 0.18, 0.25)):
    density_water = 1000.0
    f_friction = 0.02
    max_velocity = float(network_manager._net_cfg.get('max_velocity_m_s', 2.5))
    n_added = 0
    for pipe_id, pipe_cfg in network_manager.pipes.items():
        prefix = pipe_id.upper().replace('-', '_')
        m_dot_var = getattr(model, f'{prefix}_m_dot', None)
        delta_p_supply = getattr(model, f'{prefix}_delta_p_supply', None)
        if m_dot_var is None or delta_p_supply is None:
            continue
        length_m = float(pipe_cfg.get('length_m', 0) or 0)
        diameter_mm = float(pipe_cfg.get('diameter_mm', 0) or 0)
        if length_m <= 0 or diameter_mm <= 0:
            continue
        d_inner_m = diameter_mm / 1000.0 * 0.94
        area_m2 = _math.pi * (d_inner_m / 2.0) ** 2
        effective_max_flow = area_m2 * max_velocity * density_water
        k_pressure = f_friction * (length_m / d_inner_m) * (density_water / 2.0) / 1e5
        k_flow = k_pressure / ((density_water * area_m2) ** 2)

        for frac in extra_fracs:
            mi = frac * effective_max_flow
            if mi <= 0:
                continue
            setattr(model, f'{prefix}_dp_lowflow_{int(frac*1000)}',
                    pyo.Constraint(model.t, rule=(
                        lambda m, t, _mdv=m_dot_var, _dp=delta_p_supply, _mi=mi, _kf=k_flow:
                        _dp[t] >= 2.0 * _kf * _mi * _mdv[t] - _kf * _mi ** 2
                    )))
            n_added += 1
    return n_added

print("add_low_flow_tangents() defined (delta_p only -- P_pump now handled natively by pipe_pair.py's PWL pin)")


### Cell 7 — Helper: Modell bauen (ohne Solve), Review, Solve + Export

Ruft ausschließlich reale `calion/`-Funktionen (`build_model`, `export_all_results`) auf -- Physik-Nachschärfung nur via `add_low_flow_tangents` oben, kein sonstiger Nachbau der Physik. `build_scenario`/`solve_and_export` sind getrennt, damit man dazwischen `review_model()` (Cell 7b) aufrufen kann -- das gebaute Pyomo-Modell mit allen Variablen/Constraints existiert bereits VOR dem ersten Gurobi-Aufruf.

In [ ]:
# =============================================================================
# Cell 7 — build_scenario() / solve_and_export() / run_scenario(), using the
# real pipeline. Split so the model can be inspected (Cell 7b) between build
# and solve.
# =============================================================================
def build_scenario(scen_cfg_dict, label):
    tmp = _dump_yaml_tmp(scen_cfg_dict)
    inputs = _build_workflow_inputs([str(tmp)], overrides=None)
    try:
        tmp.unlink()
    except OSError:
        pass

    t0 = time.perf_counter()
    model = build_model(inputs.table, inputs.cfg, dt_h=inputs.dt_h)
    t_build = time.perf_counter() - t0

    n_tangents = add_low_flow_tangents(model, model._network_manager)
    print(f"[{label}] build={t_build:.1f}s, added {n_tangents} low-flow tangent constraints "
          f"-- NOT solved yet, safe to inspect with review_model()")
    return model, inputs


def solve_and_export(model, inputs, label, time_limit_s=1800, mip_gap=0.02, tee=False):
    solver_options = dict(inputs.cfg.get("run", {}).get("solver_options", {}))
    solver_options["TimeLimit"] = time_limit_s
    solver_options["MIPGap"] = mip_gap

    opt = pyo.SolverFactory(inputs.solver_name)
    for key, value in solver_options.items():
        opt.options[key] = value

    t1 = time.perf_counter()
    solver_result = opt.solve(model, tee=tee, warmstart=False, load_solutions=False)
    t_solve = time.perf_counter() - t1

    try:
        n_sol = len(solver_result.solution)
    except Exception:
        n_sol = 0

    status = solver_result.solver.status
    termination = solver_result.solver.termination_condition
    print(f"[{label}] solve={t_solve:.1f}s solutions={n_sol} "
          f"status={status} termination={termination}")

    if n_sol == 0:
        raise RuntimeError(
            f"[{label}] no incumbent found within {time_limit_s}s -- raise time_limit_s "
            f"or check feasibility (e.g. min_required_bar / setpoint too aggressive)."
        )

    model.solutions.load_from(solver_result)

    export_dir = str(_ROOT / "output" / "pressure_runs" / f"PUMP_STUDY_{label}" / "thermal_network_results")
    export_all_results(
        model=model,
        network_manager=getattr(model, "_network_manager", None),
        time_set=model.t,
        output_dir=export_dir,
        dt_h=inputs.dt_h,
        export_solver_files=False,
    )
    return model, inputs, export_dir


def run_scenario(scen_cfg_dict, label, time_limit_s=1800, mip_gap=0.02, tee=False):
    model, inputs = build_scenario(scen_cfg_dict, label)
    return solve_and_export(model, inputs, label, time_limit_s=time_limit_s, mip_gap=mip_gap, tee=tee)

print("build_scenario() / solve_and_export() / run_scenario() defined")


### Cell 7b — Modell-Review VOR dem Solve (Struktur + Constraints prüfen)

Baut ein Modell (hier: Szenario A) und inspiziert es symbolisch, OHNE Gurobi
aufzurufen -- Pyomo hat zu diesem Zeitpunkt bereits jede Variable/Constraint
mit ihrer exakten Formel gebaut, nur noch nicht gelöst.

Drei Ebenen:
1. **Größe**: Anzahl Variablen/Constraints/Binärvariablen insgesamt.
2. **Constraint-Familien**: jede Druck-/Pumpen-bezogene Constraint-Gruppe mit
   Anzahl Instanzen (z.B. `pressure_supply_prop_j1_to_j2: 744` = eine pro
   Zeitschritt) -- Abgleich gegen die Constraint-Liste in Implementation
   Statement Part H.4.
3. **Symbolische Formel**: die tatsächliche (unausgewertete) Gleichung für
   einen gewählten Knoten/Rohr -- zeigt echte Variablennamen und (bei den
   Tangenten-Constraints) die tatsächlichen eingesetzten Zahlenkoeffizienten,
   damit man z.B. `k_flow` gegen eine Handrechnung prüfen kann.

Für eine vollständige, durchsuchbare Kopie von allem, was Gurobi wirklich
sieht: `write_lp=True` schreibt das exakte LP-File (mit Klartext-Namen statt
x1, x2, ...) nach `output/pressure_runs/<label>_review.lp` -- in jedem
Texteditor durchsuchbar.

In [ ]:
# =============================================================================
# Cell 7b — review_model(): symbolic, pre-solve inspection
# =============================================================================
def review_model(model, focus_node="j_12", focus_pipe="j1_to_j2", write_lp=False, label="model"):
    n_vars = sum(1 for _ in model.component_data_objects(pyo.Var, active=True))
    n_bin = sum(1 for v in model.component_data_objects(pyo.Var, active=True) if v.is_binary())
    n_constr = sum(1 for _ in model.component_data_objects(pyo.Constraint, active=True))
    print(f"[{label}] n_vars={n_vars:,}  n_binary={n_bin:,}  n_constraints={n_constr:,}")

    print(f"\n[{label}] Druck-/Pumpen-Constraint-Familien (Name: #Instanzen):")
    keywords = ("pressure", "pump", "head", "delta_p")
    for con in model.component_objects(pyo.Constraint, active=True):
        name = con.name
        if any(k in name.lower() for k in keywords):
            try:
                n = sum(1 for _ in con)
            except Exception:
                n = 1
            print(f"  {name}: {n}")

    t0 = list(model.t)[0]
    print(f"\n[{label}] Symbolische Form ausgewählter Constraints bei t={t0} "
          f"(node={focus_node}, pipe={focus_pipe}):")
    pipe_prefix = focus_pipe.upper().replace('-', '_')
    candidates = [
        f"producer_{focus_node}_P_supply_setpoint",
        f"producer_{focus_node}_P_supply_floor",
        f"producer_{focus_node}_head_ub",
        f"producer_{focus_node}_head_lb",
        f"producer_{focus_node}_pump_power_agg",
        f"consumer_{focus_node}_P_min",
        f"pressure_supply_prop_{focus_pipe}",
        f"pressure_return_prop_{focus_pipe}",
        f"{pipe_prefix}_pressure_drop_supply",
        f"{pipe_prefix}_pump_power",
    ]
    for cname in candidates:
        comp = getattr(model, cname, None)
        if comp is None:
            continue
        inst = None
        for idx in (t0, (t0, 0)):
            try:
                inst = comp[idx]
                break
            except Exception:
                continue
        if inst is None:
            continue
        print(f"  {cname}:  {inst.expr}")

    if write_lp:
        lp_path = _ROOT / "output" / "pressure_runs" / f"{label}_review.lp"
        lp_path.parent.mkdir(parents=True, exist_ok=True)
        model.write(str(lp_path), io_options={"symbolic_solver_labels": True})
        print(f"\n[{label}] Vollständiges LP-File (exakt was Gurobi sieht): {lp_path}")


# Build Scenario A ONLY as far as the model object -- no solver call yet.
model_A_preview, inputs_A_preview = build_scenario(cfg_A, "A_preview")
review_model(model_A_preview, focus_node="j_12", focus_pipe="j1_to_j2", write_lp=True, label="A_preview")


### Cell 8 — Szenario A lösen (nur zentrale Pumpe am primary_producer)

In [ ]:
# =============================================================================
# Cell 8 — Solve Scenario A ("central")
# Raise TIME_LIMIT_S for a tighter MIP gap.
# =============================================================================
TIME_LIMIT_S = 1800

model_A, inputs_A, dir_A = run_scenario(cfg_A, "A_central", time_limit_s=TIME_LIMIT_S)


### Cell 9 — Szenario B lösen (zentral + zweite Pumpstation an j_13)

In [ ]:
# =============================================================================
# Cell 9 — Solve Scenario B ("central_plus_j13")
# =============================================================================
model_B, inputs_B, dir_B = run_scenario(cfg_B, "B_central_plus_j13", time_limit_s=TIME_LIMIT_S)


### Cell 10 — Extraktion: Knotendruck + Pumpenleistung (Erzeuger UND Übergabestationen)

`producer_{node}_P_pump` (Rohrreibung, aggregiert pro Pumpstation) existiert bereits im gelösten Pyomo-Modell (`network_manager.py::_link_pump_head`), wird aber vom Standard-Export NICHT herausgeschrieben -- hier direkt aus dem Modell gezogen. Seit 2026-07-27 gibt es zusätzlich `station_{node}_P_pump` pro Verbraucherknoten (Übergabestations-Differenzdruck, siehe Implementation Statement Part H) -- **beide** zusammen ergeben erst die reale Gesamt-Pumpenleistung, die gegen die installierte Kapazität (Part H, reale Wilo-Pumpen an `j_9`) verglichen werden muss. `producer_{node}_pump_head` gibt es seit 2026-07-24 nicht mehr -- Kopfdifferenz wird direkt aus P_supply - P_return berechnet.

In [ ]:
# =============================================================================
# Cell 10 — Pull node pressure + per-pump power directly from the solved
# models (these Vars exist but aren't in the standard CSV export yet).
# =============================================================================
def node_pressure_prefix(node_id):
    return node_id.upper().replace('-', '_')

def get_node_pressure(model, node_id, time_set):
    p = node_pressure_prefix(node_id)
    p_sup = getattr(model, f'{p}_pressure_supply', None)
    p_ret = getattr(model, f'{p}_pressure_return', None)
    ts = list(time_set)
    sup = [pyo.value(p_sup[t]) for t in ts] if p_sup is not None else None
    ret = [pyo.value(p_ret[t]) for t in ts] if p_ret is not None else None
    return sup, ret

def get_pump_power(model, node_id, time_set):
    '''Pipe-friction pump power aggregated at ONE producer node (BFS-attributed,
    network_manager.py::_link_pump_head). Does NOT include station_* terms --
    use get_total_pump_power for the full network-wide electrical draw.'''
    power = getattr(model, f'producer_{node_id}_P_pump', None)
    ts = list(time_set)
    return [pyo.value(power[t]) for t in ts] if power is not None else None

def get_total_pump_power(model, time_set):
    '''Sum EVERY pump-power Var in the model: producer_{node}_P_pump (pipe
    friction, one per pump station), station_{node}_P_pump (per-consumer
    transfer-station differential pressure, 2026-07-27), PLUS {NODE}_P_pump_lateral
    (real-DXF service-lateral loss, 2026-07-29 -- note this one is named with
    the node's own upper-case prefix, NOT a "station_"/"producer_" prefix,
    since it lives in thermal_node.py's own naming convention, not
    network_manager.py's). This is the number that belongs against the real
    installed pump nameplate capacity -- any one term alone under-counts the
    true electrical draw.'''
    ts = list(time_set)
    pump_var_names = [
        name for name in dir(model)
        if (
            (name.endswith('_P_pump') and (name.startswith('producer_') or name.startswith('station_')))
            or (name.endswith('_P_pump_lateral') and not name.endswith('_def'))
        )
    ]
    total = [0.0] * len(ts)
    for name in pump_var_names:
        var = getattr(model, name)
        for i, t in enumerate(ts):
            total[i] += pyo.value(var[t])
    return total, len(pump_var_names)

ALL_NODES = list(cfg["network"]["nodes"].keys())
ts_A = list(model_A.t)
ts_B = list(model_B.t)
dt_h_A = inputs_A.dt_h
dt_h_B = inputs_B.dt_h

node_pressure_A = {nid: get_node_pressure(model_A, nid, ts_A) for nid in ALL_NODES}
node_pressure_B = {nid: get_node_pressure(model_B, nid, ts_B) for nid in ALL_NODES}

pump_power_primary_A = get_pump_power(model_A, PRIMARY_PRODUCER, ts_A)
pump_power_primary_B = get_pump_power(model_B, PRIMARY_PRODUCER, ts_B)
pump_power_j13_B = get_pump_power(model_B, "j_13", ts_B)  # None in A -- j_13 has no pump there

total_pump_power_A, n_pump_vars_A = get_total_pump_power(model_A, ts_A)
total_pump_power_B, n_pump_vars_B = get_total_pump_power(model_B, ts_B)

print("Scenario A: producer_j_13_P_pump present?", getattr(model_A, "producer_j_13_P_pump", None) is not None)
print("Scenario B: producer_j_13_P_pump present?", getattr(model_B, "producer_j_13_P_pump", None) is not None)
print(f"Scenario A: {n_pump_vars_A} pump-power Vars summed into total_pump_power_A")
print(f"Scenario B: {n_pump_vars_B} pump-power Vars summed into total_pump_power_B")


### Cell 11 — Vergleichstabelle: reicht die zentrale Pumpe, was ändert die zweite?

In [ ]:
# =============================================================================
# Cell 11 — Summary comparison table
# =============================================================================
# Real installed pump nameplate capacity at j_9 ("Energiezentrale"), from
# configs/pressure/Memmingen_componenets_spec/20260729_Netzkomponenten_
# Spezifikationen.xlsx, sheet "Pumpen" -- all 5 Wilo pumps assumed able to run
# in parallel (per user confirmation 2026-07-29):
#   Dachser 1 (IL-E65/11-64BF-R1)   25.1 kW
#   Dachser 2 (IL-E65/11-64BF-R1)   25.1 kW
#   Goldi 1_alt (IL-E65/9-46BF-R1)  18.3 kW
#   Goldi 1_neu (IL-E65/220-22/2)   24.0 kW
#   Goldi 2 (IL-E65/9-46BF-R1)      18.3 kW
REAL_INSTALLED_PUMP_CAPACITY_KW = 25.1 + 25.1 + 18.3 + 24.0 + 18.3  # = 110.8 kW
print(f"Real installed pump capacity (all 5 Wilo pumps, parallel): "
      f"{REAL_INSTALLED_PUMP_CAPACITY_KW:.1f} kW")

def summarize_pump(label, power_series, dt_h):
    if power_series is None:
        return {"scenario": label, "P_pump_max_MW": None, "E_pump_MWh": None}
    return {
        "scenario": label,
        "P_pump_max_MW": max(power_series),
        "E_pump_MWh": sum(power_series) * dt_h,
    }

pump_summary = pd.DataFrame([
    summarize_pump(f"A_central: {PRIMARY_PRODUCER}", pump_power_primary_A, dt_h_A),
    summarize_pump(f"B_central_plus_j13: {PRIMARY_PRODUCER}", pump_power_primary_B, dt_h_B),
    summarize_pump("B_central_plus_j13: j_13", pump_power_j13_B, dt_h_B),
    summarize_pump("A_central: TOTAL (all producer_* + station_*)", total_pump_power_A, dt_h_A),
    summarize_pump("B_central_plus_j13: TOTAL (all producer_* + station_*)", total_pump_power_B, dt_h_B),
])
print("\n=== Pumpenauslastung (elektrische Leistung/Energie) ===")
print(pump_summary.to_string(index=False))

print("\n=== Reicht die installierte Kapazität? ===")
for label, total in [("A_central", total_pump_power_A), ("B_central_plus_j13", total_pump_power_B)]:
    peak_kw = max(total) * 1000.0
    margin_kw = REAL_INSTALLED_PUMP_CAPACITY_KW - peak_kw
    ok = "JA" if margin_kw >= 0 else "NEIN -- installierte Kapazitaet ueberschritten"
    print(f"  {label}: Spitzenlast={peak_kw:.3f} kW, installiert={REAL_INSTALLED_PUMP_CAPACITY_KW:.1f} kW, "
          f"Reserve={margin_kw:.1f} kW -> ausreichend? {ok}")

def min_pressure_margin(node_pressure_dict, pure_consumer_nodes, cfg_used):
    rows = []
    for nid in pure_consumer_nodes:
        sup, _ = node_pressure_dict.get(nid, (None, None))
        if sup is None:
            continue
        min_req = cfg_used["network"]["nodes"][nid].get("pressure", {}).get("min_required_bar", 0.0)
        rows.append({"node": nid, "P_supply_min_bar": min(sup), "min_required_bar": min_req,
                     "margin_over_floor_bar": min(sup) - min_req})
    return pd.DataFrame(rows).sort_values("margin_over_floor_bar")

print("\n=== Szenario A: Druckreserve an Verbraucherknoten (P_supply_min - min_required_bar) ===")
margin_A = min_pressure_margin(node_pressure_A, PURE_CONSUMER_NODES, cfg_A)
print(margin_A.to_string(index=False))

# In Scenario B, j_13 itself is no longer a pure consumer (has an asset) --
# exclude it from the consumer floor check there (its own pump floor applies instead).
pure_consumer_nodes_B = [n for n in PURE_CONSUMER_NODES if n != "j_13"]
print("\n=== Szenario B: Druckreserve an Verbraucherknoten (j_13 ausgenommen, hat eigene Pumpe) ===")
margin_B = min_pressure_margin(node_pressure_B, pure_consumer_nodes_B, cfg_B)
print(margin_B.to_string(index=False))


### Cell 11b — Pro-Station-Aufschlüsselung (reale Übergabestations-Anzahl)

`station_{node}_P_pump = delta_p_min_consumer_bar * m_dot_{node}` ist die
AGGREGIERTE Pumpenarbeit für den gesamten Knoten-Durchfluss -- das ist
mathematisch unabhängig davon, ob dieser Durchfluss real durch 1 oder durch
35 parallele Übergabestationen fließt (gleiches Δp pro Station, parallel
geschaltet: Summe = Δp × Gesamtdurchfluss, unabhängig von der Stationsanzahl).
`n_transfer_stations` (aus `Memmingen_pressure_stations.yaml`, demand-
proportionale Schätzung) ändert also NICHTS an der Summen-Suffizienzaussage
aus Cell 11.

Was es NEU sichtbar macht: die IMPLIZITE Pro-Station-Durchschnittslast, und
damit wie viele reale Stationen sich hinter einem einzigen
`min_required_bar`-Wert verstecken. Ein Knoten mit `n_transfer_stations=35`
(j_12) hat den gleichen validierten Mindestdruck wie einer mit 2 (j_2) --
die Modellannahme ist für j_12 am wenigsten individuell abgesichert, weil
sie für 35 reale Gebäude gleichzeitig gelten müsste, nicht nur für eines.

In [ ]:
# =============================================================================
# Cell 11b — Per-station average pump power/flow, using n_transfer_stations
# from the config (demand-proportional estimate, see YAML header comment).
# =============================================================================
def get_node_total_pump_power(model, node_id, time_set):
    '''Sum this ONE node's own producer_*_P_pump (if it's a pump station),
    station_*_P_pump (if it has a transfer-station Δp term), PLUS
    {NODE}_P_pump_lateral (real-DXF lateral loss, already scaled by
    n_transfer_stations inside thermal_node.py -- see that module's
    P_pump_lateral_def constraint) -- i.e. the total electrical pump draw
    attributable to this node specifically (unlike get_total_pump_power,
    which sums the whole network).'''
    ts = list(time_set)
    total = [0.0] * len(ts)
    found = False
    _node_prefix = node_id.upper().replace('-', '_')
    for prefix in (f"producer_{node_id}_P_pump", f"station_{node_id}_P_pump",
                   f"{_node_prefix}_P_pump_lateral"):
        var = getattr(model, prefix, None)
        if var is None:
            continue
        found = True
        for i, t in enumerate(ts):
            total[i] += pyo.value(var[t])
    return total if found else None


def per_station_table(model, cfg_used, time_set, dt_h, label):
    ts = list(time_set)
    rows = []
    for nid, ncfg in cfg_used["network"]["nodes"].items():
        n_stations = ncfg.get("n_transfer_stations", 0)
        node_power = get_node_total_pump_power(model, nid, ts)
        if node_power is None or n_stations == 0:
            continue
        peak_kw = max(node_power) * 1000.0
        rows.append({
            "scenario": label, "node": nid, "n_transfer_stations": n_stations,
            "node_P_pump_peak_kW": peak_kw,
            "per_station_avg_peak_kW": peak_kw / n_stations,
            "node_E_pump_MWh": sum(node_power) * dt_h,
        })
    return pd.DataFrame(rows).sort_values("n_transfer_stations", ascending=False)


print("=== Szenario A: pro Knoten, pro Station (sortiert nach n_transfer_stations) ===")
station_table_A = per_station_table(model_A, cfg_A, ts_A, dt_h_A, "A_central")
print(station_table_A.to_string(index=False))

print("\n=== Szenario B: pro Knoten, pro Station ===")
station_table_B = per_station_table(model_B, cfg_B, ts_B, dt_h_B, "B_central_plus_j13")
print(station_table_B.to_string(index=False))

print("\n=== Aggregationsrisiko: Knoten mit vielen Stationen UND knapper Druckreserve ===")
risk_A = margin_A.merge(station_table_A[["node", "n_transfer_stations"]], on="node", how="left")
print(risk_A.sort_values("margin_over_floor_bar").to_string(index=False))

# --- Differential-pressure margin, the constraint the lateral-loss term
# ACTUALLY tightens (P_supply - P_return >= delta_p_min_consumer_bar +
# lateral_dp_extra[t]) -- separate from margin_A/B above, which only checks
# the unrelated absolute P_supply >= min_required_bar floor.
def lateral_dp_margin(model, cfg_used, time_set, label):
    ts = list(time_set)
    dp_station_bar = float(cfg_used["network"].get("delta_p_min_consumer_bar", 0.7))
    rows = []
    for nid, ncfg in cfg_used["network"]["nodes"].items():
        prefix = nid.upper().replace('-', '_')
        p_sup = getattr(model, f'{prefix}_pressure_supply', None)
        p_ret = getattr(model, f'{prefix}_pressure_return', None)
        lat_extra = getattr(model, f'{prefix}_lateral_dp_extra', None)
        if p_sup is None or p_ret is None:
            continue
        diffs = [pyo.value(p_sup[t]) - pyo.value(p_ret[t]) for t in ts]
        required = [dp_station_bar + (pyo.value(lat_extra[t]) if lat_extra is not None else 0.0) for t in ts]
        margins = [d - r for d, r in zip(diffs, required)]
        rows.append({
            "scenario": label, "node": nid,
            "lateral_length_m": ncfg.get("pressure", {}).get("lateral_length_m"),
            "min_diff_margin_bar": min(margins),
        })
    return pd.DataFrame(rows).sort_values("min_diff_margin_bar")


print("\n=== Differenzdruck-Marge (P_supply-P_return) inkl. Zuleitungsverlust, Szenario A ===")
print(lateral_dp_margin(model_A, cfg_A, ts_A, "A_central").to_string(index=False))


### Cell 12 — Plots: Druckprofil entlang der Trasse + Pumpenleistung über die Zeit

In [ ]:
# =============================================================================
# Cell 12 — Plots
# =============================================================================
# Real trunk to j_13 in the CURRENT topology (updated 2026-07-29 when j_9
# became the root): j_9 -> j_10 -> j_11 -> j_13 -> j_14. j_1 is now a
# dead-end leaf on a DIFFERENT branch (j_9->j_3->j_2->j_1) and is not on
# this path anymore -- don't hardcode a trunk without checking the pipes:
# section in the YAML first if you change the topology again.
TRUNK = [PRIMARY_PRODUCER, "j_10", "j_11", "j_13", "j_14"]
MIN_REQUIRED_BAR_REF = cfg["network"]["nodes"]["j_14"].get("pressure", {}).get("min_required_bar", 0.0)

peak_idx_A = int(np.argmax(pump_power_primary_A)) if pump_power_primary_A else 0
peak_idx_B = int(np.argmax(pump_power_primary_B)) if pump_power_primary_B else 0

fig, ax = plt.subplots(figsize=(9, 5))
for label, node_pressure, peak_idx, style in [
    (f"A_central ({PRIMARY_PRODUCER} only)", node_pressure_A, peak_idx_A, "-o"),
    ("B_central_plus_j13", node_pressure_B, peak_idx_B, "-s"),
]:
    y = [node_pressure[n][0][peak_idx] if node_pressure[n][0] else np.nan for n in TRUNK]
    ax.plot(TRUNK, y, style, label=label)
ax.axhline(MIN_REQUIRED_BAR_REF, color="red", ls="--", lw=1, label=f"min_required_bar={MIN_REQUIRED_BAR_REF}")
ax.set_ylabel("P_supply [bar]")
ax.set_title(f"Druckprofil entlang {'->'.join(TRUNK)} (Stunde mit Peak-Pumpleistung)")
ax.legend()
fig.tight_layout()
fig.savefig(_ROOT / "output" / "pressure_runs" / "pump_study_pressure_profile.png", dpi=150)
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(pump_power_primary_A, label=f"P_pump {PRIMARY_PRODUCER} (Szenario A)", alpha=0.8)
ax2.plot(pump_power_primary_B, label=f"P_pump {PRIMARY_PRODUCER} (Szenario B)", alpha=0.8)
if pump_power_j13_B is not None:
    ax2.plot(pump_power_j13_B, label="P_pump j_13 (Szenario B)", alpha=0.8)
ax2.plot(total_pump_power_A, label="TOTAL (Szenario A, inkl. station_*)", ls="--", alpha=0.8)
ax2.plot(total_pump_power_B, label="TOTAL (Szenario B, inkl. station_*)", ls="--", alpha=0.8)
ax2.axhline(REAL_INSTALLED_PUMP_CAPACITY_KW / 1000.0, color="red", ls=":", lw=1.5,
            label=f"installierte Kapazitaet ({REAL_INSTALLED_PUMP_CAPACITY_KW:.0f} kW)")
ax2.set_ylabel("MW_el")
ax2.set_title("Pumpenleistung über das Zeitfenster (inkl. reale installierte Kapazitaet)")
ax2.legend(fontsize=8)
fig2.tight_layout()
fig2.savefig(_ROOT / "output" / "pressure_runs" / "pump_study_power_timeseries.png", dpi=150)
plt.show()

print("CELL 12 OK -- Plots gespeichert unter output/pressure_runs/pump_study_*.png")


### Interpretation

- **Reicht die zentrale Pumpe allein (Szenario A)?** Zwei Ebenen, beide in
  Cell 11:
  1. *Druck*: `margin_A` -- ist `margin_over_floor_bar` an jedem Knoten >= 0,
     ist der konfigurierte Mindestdruck überall erfüllt (das Modell hätte
     sonst gar nicht gelöst, da es eine harte Nebenbedingung ist -- ein
     Infeasible/no incumbent in Cell 8 wäre das direkte Signal für
     "reicht nicht").
  2. *Elektrische Kapazität*: der neue "Reicht die installierte Kapazität?"
     Block -- vergleicht `total_pump_power` (ALLE `producer_*`/`station_*`
     Terme zusammen, nicht nur die eine Pumpstation) gegen die echten
     110.8 kW aus dem Wilo-Datenblatt.
- **Was ändert die zweite Pumpstation (Szenario B)?** Vergleiche
  `pump_summary`: sinkt `E_pump_MWh` am `PRIMARY_PRODUCER`, während `j_13`
  einen Teil übernimmt? Das zeigt die tatsächliche Entlastung der zentralen
  Pumpe durch die lokale Einspeisung + eigenen Druck-Floor am Südast.
- **Warum ist die Druckkurve nirgends "beliebig" hoch?** Weil
  `network.physics.pressure_regularization: true` in der YAML aktiv ist
  (siehe Implementation Statement Part H.7) -- ohne das würden Knoten
  downstream von einem fixen Erzeuger bei einem willkürlichen Solver-Wert
  landen, nicht beim physikalisch echten.
- **Was bringt `n_transfer_stations` (Cell 11b)?** Ändert die
  Suffizienzaussage aus Cell 11 NICHT (Summenleistung ist stationszahl-
  unabhängig, siehe Cell 11b-Intro) -- macht aber sichtbar, wo die
  `min_required_bar`-Annahme am dünnsten abgesichert ist: j_12 (35 geschätzte
  Stationen) und j_3 (31) tragen den größten Teil der realen 174 Stationen
  hinter jeweils nur 1-3 V_x-Spalten. Die Zahlen selbst sind eine
  demand-proportionale Schätzung (siehe `Memmingen_pressure_stations.yaml`
  Kopfkommentar für Methodik + Grenzen), keine verifizierte Zählung pro
  Knoten -- für eine echte DXF-Rekonstruktion pro Zweig müsste man den
  Pumpenleitungsgraphen matchen (analog zur früheren Durchmesser-Kalibrierung
  in project_memmingen_dxf_crosscheck), was bei ~58% zusammenhängendem
  Trassenanteil in der DXF nur mit relevanter Unsicherheit pro Zweig ginge.
- **Was bringt `pressure.lateral_length_m` (die neue Zeile in Cell 11b)?**
  ANDERS als `n_transfer_stations`: das ändert die gelöste Physik wirklich --
  vorher hatte jeder Knoten NUR den stationsinternen 0.6 bar Verlust, jetzt
  zusätzlich einen konvex-linearisierten, durchflussabhängigen Verlust für
  die reale (DXF-vermessene, bootstrap-geschätzte) Zuleitung dorthin. Die
  Gesamt-Suffizienzaussage (110.8 kW installiert) bleibt unangetastet (der
  Zusatzverlust ist klein, ~3.0->3.2 kW Spitze in Szenario A) -- die neue
  `lateral_dp_margin`-Tabelle am Ende von Cell 11b zeigt aber die tatsächlich
  verschärfte Differenzdruck-Nebenbedingung (`P_supply-P_return >=
  delta_p_min_consumer_bar + lateral_dp_extra[t]`) pro Knoten, getrennt von
  der unveränderten `min_required_bar`-Prüfung in `margin_A`/`margin_B`.
  Standardmäßig AUS ohne `pressure.lateral_length_m` in der YAML -- siehe
  memory project_memmingen_pump_pressure_study für die volle Herleitung
  (warum keine Binärvariablen nötig sind, wie die Bootstrap-Schätzung
  funktioniert, bekannte Grenzen).
- **Eigene Pumpe hinzufügen?** Siehe Implementation Statement Part H.8 --
  einfach `assets: [...]` an einem Knoten in `Memmingen_pressure.yaml`
  setzen, optional `pressure.setpoint_bar`. Kein Codeeingriff nötig, solange
  es eine echte Wärmeerzeugungs-Anlage ist (reine Booster-Pumpe ohne
  Wärmeasset braucht laut Part H.8 einen Codeeingriff).
- **Wichtig, falls du den Standort/`primary_producer` nochmal änderst:**
  `network.nodes.<id>.pressure.{setpoint_bar,min_required_bar}` wurde bis
  2026-07-29 campaign-weit von `model_finalizer.py` stillschweigend verworfen
  (fehlte in dessen Feld-Whitelist) -- jeder Nicht-Standard-Wert landete beim
  Code-Default statt beim konfigurierten Wert, ohne Fehler. Jetzt gefixt und
  verifiziert (Implementation Statement Part H.2) -- falls ein Wert wieder
  "nicht ankommt", ist das der erste Verdächtige.
- **Nächste Schritte:** TIME_LIMIT_S erhöhen für einen engeren MIP-Gap, das
  Zeitfenster in Cell 3 auf das ganze Jahr oder einen anderen Extremmonat
  ausweiten, oder `PUMP_SETPOINT_J13_BAR` / `min_required_bar` in der YAML
  gegen weitere echte Anschluss-/Pumpendaten ersetzen.
